In [2]:
import polars as pl
from pathlib import Path

base = Path("/Users/macbook/ProyectosLocales/PrecioLuz/datos")

# carpetas tipo 20XXlimpio
carpetas = sorted([
    p for p in base.iterdir()
    if p.is_dir() and p.name.endswith("limpio") and p.name.startswith("20")
])

dfs = []
for c in carpetas:
    dfs.append(pl.read_csv(str(c / "*.csv")))

df_total = pl.concat(dfs)

# limpiar
df_total = df_total.unique(subset=["Timestamp"])
df_total = df_total.sort("Timestamp")

# guardar EXACTAMENTE donde tenías
df_total.write_csv("/Users/macbook/ProyectosLocales/PrecioLuz/datos/precio_pvpc.csv")

print("Registros:", df_total.height)

df_dias = df_total.with_columns(
    pl.col("Timestamp").str.slice(0, 10).alias("fecha")
)

print("Días:", df_dias.select(pl.col("fecha").n_unique()).item())

saltos = df_dias.with_columns(
    pl.col("fecha").str.to_date()
).select("fecha").unique().sort("fecha").with_columns(
    pl.col("fecha").diff().alias("d")
)

print(saltos.filter(pl.col("d") > pl.duration(days=1)))

horas = df_dias.group_by("fecha").count()
print(horas.filter(pl.col("count") != 24))

Registros: 96432
Días: 4018
shape: (0, 2)
┌───────┬──────────────┐
│ fecha ┆ d            │
│ ---   ┆ ---          │
│ date  ┆ duration[μs] │
╞═══════╪══════════════╡
└───────┴──────────────┘
shape: (22, 2)
┌────────────┬───────┐
│ fecha      ┆ count │
│ ---        ┆ ---   │
│ str        ┆ u32   │
╞════════════╪═══════╡
│ 2015-10-25 ┆ 25    │
│ 2024-03-31 ┆ 23    │
│ 2015-03-29 ┆ 23    │
│ 2020-10-25 ┆ 25    │
│ 2022-03-27 ┆ 23    │
│ …          ┆ …     │
│ 2021-10-31 ┆ 25    │
│ 2025-10-26 ┆ 25    │
│ 2017-03-26 ┆ 23    │
│ 2016-03-27 ┆ 23    │
│ 2019-03-31 ┆ 23    │
└────────────┴───────┘


/var/folders/hx/0zmhrp2d37x4jg59pl9ktf0r0000gn/T/ipykernel_48815/2174147592.py:41: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  horas = df_dias.group_by("fecha").count()
